In [0]:
%run ../config/config

In [0]:
# Imports
import logging
import socket
import os
import sys
import time

sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger, log_dataframe_info
import pandas as pd

In [0]:
# Logger inference
logger = MLOpsLogger(
    name='05_inference',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '05_inference'
    }
)

In [0]:
# =============================================================================
# PARÁMETROS DE EJECUCIÓN MULTI-MES (desde config centralizado)
# =============================================================================

from utils.month_delta import month_delta
RUN_HISTORICAL = first_run  # Controlado por bandera FIRST_RUN del config
if RUN_HISTORICAL and num_meses_historicos > 0:
    # Generar lista de meses HISTÓRICOS (sin incluir el actual)
    meses_historicos_lista = []
    mes_tmp = mes_vta

    for i in range(num_meses_historicos):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)

    # Invertir para procesar desde el más antiguo al más reciente
    meses_historicos_lista.reverse()

    # Agregar el mes actual al final
    meses_a_procesar = meses_historicos_lista + [mes_vta]

    logger.info("="*80)
    logger.info("MODO HISTÓRICO ACTIVADO (FIRST_RUN)")
    logger.info("="*80)
    logger.info(f" Se procesarán {len(meses_a_procesar)} meses para inferencia:")
    logger.info(f"  Históricos: {meses_historicos_lista}")
    logger.info(f"  Actual: {mes_vta}")
    logger.info("="*80)
else:
    meses_a_procesar = [mes_vta]
    logger.info(f"Procesando únicamente el mes actual: {mes_vta}")

In [0]:
import sasindb
from sasindb.mla import Model
from pyspark.sql import DataFrame, SQLContext, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
import time

# =============================================================================
# LOOP PRINCIPAL POR CADA MES
# =============================================================================
resultados_inferencia = []
for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("="*80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("="*80)

    full_table_name_mt = TABLE_MASTER
    try:
        if use_segments:
            # MODO CON SEGMENTOS
            logger.info("Leyendo master table...")
            df_master = spark.sql(f"SELECT * FROM {full_table_name_mt} WHERE codmes = {mes_actual_proceso}")
            total_records = df_master.count()
            logger.info(f"Total de registros en master table: {total_records:,}")

            scored_dataframes = []

            for seg_config in segments_config:
                segment_id = seg_config["segment_id"]
                segment_name = seg_config["name"]
                segment_filter = seg_config["filter"]
                model_name = seg_config["model_table"]

                logger.info(f"PROCESANDO SEGMENTO {segment_id}: {segment_name}")

                df_segment = df_master.filter(segment_filter).withColumn("segment_id", lit(segment_id))
                count_segment = df_segment.count()
                logger.info(f"Registros en segmento {segment_id}: {count_segment:,}")

                # Guardar segmento
                segment_table = f"{catalog_name}.{schema_name}.bcp_segmento_{segment_id}_{mes_actual_proceso}"
                df_segment.write.mode("overwrite").saveAsTable(segment_table)

                # Aplicar modelo
                model_table = f"{catalog_name}.{SCHEMA_MODELS}.{model_name}"
                modelds = spark.table(model_table)
                batch_model = Model.create(modelds, df_segment)
                batch_model.setDBMaxText(32767)
                scored = batch_model.score()

                # Guardar scoring temporal
                scored_table = f"{catalog_name}.{schema_name}.bcp_scored_seg{segment_id}_{mes_actual_proceso}"
                scored.write.mode("overwrite").saveAsTable(scored_table)

                scored_dataframes.append(scored)

            # Consolidar segmentos
            df_final = scored_dataframes[0]
            for df in scored_dataframes[1:]:
                df_final = df_final.union(df)

        else:
            # MODO SIN SEGMENTOS
            df_master = spark.sql(f"SELECT * FROM {full_table_name_mt} WHERE codmes = {mes_actual_proceso}")
            total_records = df_master.count()
            model_name = TABLE_MODEL
            model_table = f"{catalog_name}.{SCHEMA_MODELS}.{model_name}"
            modelds = spark.table(model_table)
            batch_model = Model.create(modelds, df_master)
            batch_model.setDBMaxText(32767)
            df_final = batch_model.score()

        # FIX - El scoreo altera la tabla de input: Mantener solo las columnas generadas por .score()
        #df_final = df_final.select('codclavepartycli', 'P_def30_121') # codmes no es necesario, el loop recorre 1 a 1
        df_final = df_final.select('codclavepartycli', 'numpd')        
        df_final = df_master.join(df_final, 'codclavepartycli', 'left')

        # numpd, numxb, numsc son nombres necesarios para la implementación
        #df_final = df_final.withColumnRenamed('P_def30_121', 'numpd') # 'P_def30_121' puede parametrizarse en config.ipynb
        df_final = df_final.withColumn("numxb", F.log(F.col("numpd") / (1 - F.col("numpd"))))
        df_final = df_final.withColumn(
            "numsc",
            F.when(
                F.col("numxb").isNotNull(),
                F.greatest( F.lit(3), F.round(174.25 - 57.708 * F.col("numxb"), 1) )
            )
        )

        # SIEMPRE: Eliminar mes existente antes de insertar (evitar duplicados)
        logger.info(f"Sobrescribiendo datos existentes para codmes={mes_actual_proceso} usando replaceWhere")

        # Verificar si la tabla existe (para logging consistente)
        try:
            # Safer way to check for table existence with 3-part namespace
            spark.read.table(output_table).limit(0)
            table_exists = True
        except:
            table_exists = False

        if table_exists:
            logger.info(f"Tabla {output_table} existe. Usando replaceWhere para mes {mes_actual_proceso}...")
            # Usar overwrite con replaceWhere para asegurar atomicidad
            df_final.write \
                .format("delta") \
                .mode("overwrite") \
                .option("replaceWhere", f"codmes = {mes_actual_proceso}") \
                .saveAsTable(output_table)
            logger.info("Datos sobrescritos exitosamente")
        else:
            # Crear tabla por primera vez
            logger.info(f"Creando tabla {output_table} por primera vez...")
            df_final.write.mode("overwrite").format("delta").saveAsTable(output_table)

        final_count = df_final.count()
        logger.info(f"✓ Inferencia completada para mes {mes_actual_proceso}: {final_count:,} registros")

        resultados_inferencia.append({
            'mes': mes_actual_proceso,
            'registros': final_count,
            'exitoso': True
        })

    except Exception as e:
        logger.error(f"Error procesando mes {mes_actual_proceso}: {str(e)}")
        resultados_inferencia.append({
            'mes': mes_actual_proceso,
            'registros': 0,
            'exitoso': False,
            'error': str(e)
        })
        continue

# =============================================================================
# RESUMEN FINAL
# =============================================================================
logger.info("")
logger.info("="*80)
logger.info("RESUMEN FINAL DE INFERENCIA")
logger.info("="*80)
exitosos = sum(1 for r in resultados_inferencia if r['exitoso'])
logger.info(f"Total meses procesados: {len(resultados_inferencia)}")
logger.info(f"Exitosos: {exitosos}")
logger.info(f"Fallidos: {len(resultados_inferencia) - exitosos}")
logger.info("="*80)

#Validaciones

In [0]:
from pyspark.sql import functions as F
df_out = spark.read.table(OUTPUT_TABLE).filter(F.col("codmes")==202512)
df_out.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("numpd").isNull(),1).otherwise(0)).alias("pd_nulos"),
    F.min("numpd").alias("numpd_min"), F.max("numpd").alias("numpd_max"), F.mean("numpd").alias("numpd_prom"),
).show()

In [0]:
from pyspark.sql import functions as F

df_mio = (spark.read.table(OUTPUT_TABLE)
          .filter(F.col("codmes")==202512)
          .select("codmes","codclaveunicocli", F.col("numpd").alias("numpd_mio")))

df_modelador = (spark.table("catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_cor_copy")
                .filter(F.col("codmes")==202512)
                .select("codmes","codclaveunicocli", F.col("PD_NUEVO").alias("pd_modelador")))

print("mio:", df_mio.count(), " | Modelador:", df_modelador.count())

cmp = (df_mio.join(df_modelador, ["codmes","codclaveunicocli"], "inner")
       .withColumn("dif", F.abs(F.col("numpd_mio")-F.col("pd_modelador"))))

cmp.select(
    F.count("*").alias("n"),
    F.mean("dif").alias("dif_prom"),
    F.max("dif").alias("dif_max"),
    F.sum(F.when(F.col("dif") <= 1e-6, 1).otherwise(0)).alias("coinciden"),
    F.sum(F.when(F.col("dif") <= 1e-4, 1).otherwise(0)).alias("coinciden_1e4"),
).show()

In [0]:
spark.read.table(OUTPUT_TABLE).groupby("codmes").count().show()

In [0]:
display(
    spark.sql(
        "DESCRIBE catalog_cemm_expl_bcp_prod.bcp_expl_007.bcp_tb_score_pd_bhv_pyme_dev"
    )
)

In [0]:
display(
    spark.sql(
        "DESCRIBE catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_cor_copy"
    )
)

In [0]:
# Lectura de las tablas
df_lhcl = spark.table(
    "catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_cor_copy"
).filter(F.col("codmes") == 202512)
df_cemm = spark.table(
    "catalog_cemm_expl_bcp_prod.bcp_expl_007.bcp_tb_score_pd_bhv_pyme_dev"
).filter(F.col("codmes") == 202512)

# Promedios en la tabla LHCL
promedios_lhcl = df_lhcl.agg(
    F.avg("PD_NUEVO").alias("avg_PD_NUEVO_Modelador"),
    F.avg("XB_NUEVO").alias("avg_XB_NUEVO_Modelador"),
    F.avg("SC_NUEVO").alias("avg_SC_NUEVO_Modelador")
)

# Promedios en la tabla CEMM
promedios_cemm = df_cemm.agg(
    F.avg("numpd").alias("avg_numpd"),
    F.avg("numxb").alias("avg_XB"),
    F.avg("numscore").alias("avg_numscore")
)

# Mostrar resultados
display(promedios_lhcl)
display(promedios_cemm)

In [0]:
# Validación: Tabla Score (mía) vs Tabla de Modelador

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

# =============================================================================
# PARÁMETROS
# =============================================================================
TABLA_MIA       = "catalog_cemm_expl_bcp_prod.bcp_expl_007.bcp_tb_score_pd_bhv_pyme_dev"
TABLA_MODELADOR = "catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_cor_copy"
CODMES          = 202512

# Mapeo de columnas de score: nombre_en_mi_tabla -> nombre_en_tabla_modelador
SCORE_MAP = {
    "numpd":   "PD_NUEVO",
    "numscore": "SC_NUEVO",
}

THRESHOLD = 1e-6  # tolerancia para considerar que dos valores coinciden

# Paso 1 – Verificar conteos y existencia de columnas
df_mia = spark.read.table(TABLA_MIA).filter(F.col("codmes") == CODMES)
df_modelador = spark.table(TABLA_MODELADOR).filter(F.col("codmes") == CODMES)

print(f"Registros mía: {df_mia.count():,}")
print(f"Registros Modelador: {df_modelador.count():,}")
print(f"\nColumnas mía: {len(df_mia.columns)}")
print(f"Columnas Modelador: {len(df_modelador.columns)}")

# Paso 2 – Comparar scores (PD / PD_CAL)
for col_mia, col_modelador in SCORE_MAP.items():
    if col_mia not in df_mia.columns:
        print(f"⚠️ {col_mia} no existe en mi tabla, se omite")
        continue
    if col_modelador not in df_modelador.columns:
        print(f"⚠️ {col_modelador} no existe en tabla Modelador, se omite")
        continue

    a = df_mia.select("codmes", "codclaveunicocli", F.col(col_mia).alias("v_mia"))
    b = df_modelador.select("codmes", "codclaveunicocli", F.col(col_modelador).alias("v_modelador"))

    cmp = (a.join(b, ["codmes", "codclaveunicocli"], "inner")
           .withColumn("dif", F.abs(F.col("v_mia") - F.col("v_modelador"))))

    res = cmp.select(
        F.count("*").alias("n"),
        F.mean("dif").alias("dif_prom"),
        F.max("dif").alias("dif_max"),
        F.sum(F.when(F.col("dif") <= THRESHOLD, 1).otherwise(0)).alias("coinciden"),
    ).collect()[0]

    pct = 100 * res["coinciden"] / res["n"] if res["n"] else 0
    print(f"\n=== {col_mia} vs {col_modelador} ===")
    print(f"  n={res['n']:,} | coinciden={res['coinciden']:,} ({pct:.4f}%)")
    print(f"  dif_prom={res['dif_prom']:.2e} | dif_max={res['dif_max']:.2e}")

# Paso 3 – Comparar TODAS las variables comunes (features + scores)
cols_keys = ["codmes", "codclaveunicocli"]
excluir = set(cols_keys + ["codclavepartycli", "codinternocomputacional"])

# variables comunes a ambas tablas (mismo nombre)
comunes = [c for c in df_mia.columns if c in df_modelador.columns and c not in excluir]
print(f"Variables comunes a comparar: {len(comunes)}")

n_mia = df_mia.count()
filas = []

for c in comunes:
    dtype = dict(df_mia.dtypes)[c]
    a = df_mia.select(*cols_keys, F.col(c).alias("v_mia"))
    b = df_modelador.select(*cols_keys, F.col(c).alias("v_modelador"))
    j = a.join(b, cols_keys, "inner")

    if dtype in ("string"):
        eq = j.withColumn("eq", F.when(
            (F.col("v_mia").isNull() & F.col("v_modelador").isNull()) |
            (F.col("v_mia") == F.col("v_modelador")), 1).otherwise(0))
    else:
        eq = j.withColumn("eq", F.when(
            (F.col("v_mia").isNull() & F.col("v_modelador").isNull()) |
            (F.abs(F.col("v_mia") - F.col("v_modelador")) <= THRESHOLD), 1).otherwise(0))

    agg = eq.agg(
        F.count("*").alias("n"),
        F.sum("eq").alias("iguales")
    ).collect()[0]

    n = agg["n"] or 0
    iguales = agg["iguales"] or 0
    pct = 100 * iguales / n if n else 0
    filas.append({
        "variable": c,
        "n": n,
        "iguales": iguales,
        "porc_equiv": round(pct, 4),
        "validacion": "ok" if pct >= 99.0 else "falla"
    })

resumen = (pd.DataFrame(filas)
           .sort_values("porc_equiv", ascending=True)
           .reset_index(drop=True))
print(resumen.to_string())

# Paso 4 – Resumen final
total    = len(resumen)
ok       = (resumen["validacion"] == "ok").sum()
fallan   = total - ok

print("=" * 50)
print(f"RESUMEN DE VALIDACIÓN (codmes={CODMES})")
print("=" * 50)
print(f"Variables comparadas: {total}")
print(f"  ✓ OK   (>=99%): {ok}")
print(f"  X Falla (<99%): {fallan}")

if fallan > 0:
    print("\nVariables que fallan:")
    print(resumen[resumen["validacion"] == "falla"].to_string(index=False))
else:
    print("\n✓✓✓ Todas las variables y scores coinciden con la tabla de Modelador.")